In [1]:
!pip install ultralytics supervision matplotlib seaborn tifffile shapely opencv-python -q

In [2]:
import os, json, glob, shutil, cv2, tifffile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from PIL import Image
from shapely import wkt
from pathlib import Path
from tqdm import tqdm

# ── Class config (same as SAM2 notebook) ─────────────────────────────────────
SEG_CLASSES  = ['background', 'intact', 'damaged', 'destroyed']
NUM_CLASSES  = 4

# YOLO seg uses 0-indexed, background is NOT a YOLO class (YOLO ignores BG)
# We only annotate the 3 building classes for YOLO polygon seg format
YOLO_CLASSES = ['intact', 'damaged', 'destroyed']   # class 0, 1, 2

XBD_TO_YOLO = {
    'no-damage':    0,   # intact
    'minor-damage': 1,   # damaged (merged)
    'major-damage': 1,   # damaged (merged)
    'destroyed':    2,   # destroyed
}

CLASS_COLORS = {
    0: (0,   0,   0),    # background — black
    1: (0,   200, 0),    # intact     — green
    2: (255, 165, 0),    # damaged    — orange
    3: (220, 0,   0),    # destroyed  — red
}

# Paths
BASE_DATA_DIR = r"E:\UTS\CNN and Deep Learning\Assignment 3\42028-DLCNN\data\xView2\geotiffs"
YOLO_DS_DIR   = r"E:\UTS\CNN and Deep Learning\Assignment 3\42028-DLCNN\xbd_yolo"
IMG_SIZE      = 640   # YOLO standard

TRAIN_DIRS = [os.path.join(BASE_DATA_DIR, "tier1"),
              os.path.join(BASE_DATA_DIR, "tier3")]
VAL_DIR    = os.path.join(BASE_DATA_DIR, "hold")
TEST_DIR   = os.path.join(BASE_DATA_DIR, "test")

print("Config ready.")

Config ready.


In [3]:
def polygon_to_yolo_seg(coords, img_w, img_h):
    """
    Convert polygon coords (N,2) to YOLO seg format:
    normalised flat list [x1/W, y1/H, x2/W, y2/H, ...]
    """
    pts = coords[:, :2].astype(np.float32)
    pts[:, 0] = np.clip(pts[:, 0] / img_w, 0.0, 1.0)
    pts[:, 1] = np.clip(pts[:, 1] / img_h, 0.0, 1.0)
    return pts.flatten().tolist()


def convert_split(data_dirs, split_name, output_root, img_size=IMG_SIZE):
    """
    Convert one split (train/val/test) of xBD to YOLO seg format.
    Uses POST-disaster image as input (damage is visible post-event).
    """
    img_out = os.path.join(output_root, "images", split_name)
    lbl_out = os.path.join(output_root, "labels", split_name)
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    dirs = data_dirs if isinstance(data_dirs, list) else [data_dirs]
    label_files = []
    for d in dirs:
        label_files.extend(
            glob.glob(os.path.join(d, 'labels', '*post_disaster.json'))
        )

    skipped = 0
    for lp in tqdm(label_files, desc=f"Converting {split_name}"):
        base      = os.path.basename(lp).replace('_post_disaster.json', '')
        post_path = os.path.join(os.path.dirname(lp).replace('labels', 'images'),
                                 f'{base}_post_disaster.tif')

        if not os.path.exists(post_path):
            skipped += 1
            continue

        # ── Load & resize post-disaster image ────────────────────────────────
        img = np.array(tifffile.imread(post_path))
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        if img.max() <= 1.0:
            img = (img * 255).astype(np.uint8)
        else:
            img = img.astype(np.uint8)

        orig_h, orig_w = img.shape[:2]
        img_resized = cv2.resize(img, (img_size, img_size),
                                 interpolation=cv2.INTER_LINEAR)

        # Save image as PNG
        img_save_path = os.path.join(img_out, f"{base}.png")
        cv2.imwrite(img_save_path, cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR))

        # ── Parse label JSON → YOLO seg txt ──────────────────────────────────
        with open(lp) as f:
            label = json.load(f)

        lines = []
        for feat in label['features']['xy']:
            damage = feat['properties'].get('subtype', 'no-damage')
            if damage not in XBD_TO_YOLO:
                continue
            yolo_cls = XBD_TO_YOLO[damage]

            try:
                poly   = wkt.loads(feat['wkt'])
                coords = np.array(poly.exterior.coords, dtype=np.float32)
            except Exception:
                continue

            if len(coords) < 3:
                continue

            # Scale polygon coords from original image size to IMG_SIZE
            coords[:, 0] = coords[:, 0] * (img_size / orig_w)
            coords[:, 1] = coords[:, 1] * (img_size / orig_h)

            norm_pts = polygon_to_yolo_seg(coords, img_size, img_size)
            if len(norm_pts) < 6:   # need at least 3 points
                continue

            pts_str = " ".join(f"{v:.6f}" for v in norm_pts)
            lines.append(f"{yolo_cls} {pts_str}")

        # Write label file (even if empty — YOLO needs it)
        lbl_save_path = os.path.join(lbl_out, f"{base}.txt")
        with open(lbl_save_path, 'w') as f:
            f.write("\n".join(lines))

    print(f"  {split_name}: {len(label_files)-skipped} scenes, {skipped} skipped")


# ── Run conversion for all splits ────────────────────────────────────────────
print("Converting xBD → YOLO segmentation format...")
convert_split(TRAIN_DIRS, "train", YOLO_DS_DIR)
convert_split(VAL_DIR,    "val",   YOLO_DS_DIR)
convert_split(TEST_DIR,   "test",  YOLO_DS_DIR)
print("Conversion complete.")

Converting xBD → YOLO segmentation format...


Converting train: 100%|██████████| 9168/9168 [10:40<00:00, 14.31it/s]


  train: 9168 scenes, 0 skipped


Converting val: 100%|██████████| 933/933 [00:50<00:00, 18.62it/s]


  val: 933 scenes, 0 skipped


Converting test: 100%|██████████| 933/933 [00:51<00:00, 18.15it/s]

  test: 933 scenes, 0 skipped
Conversion complete.


In [4]:
yaml_content = f"""
path: {YOLO_DS_DIR.replace(chr(92), '/')}
train: images/train
val:   images/val
test:  images/test

nc: 3
names: {YOLO_CLASSES}
"""

yaml_path = os.path.join(YOLO_DS_DIR, "xbd.yaml")
with open(yaml_path, 'w') as f:
    f.write(yaml_content.strip())

print(f"Dataset YAML written → {yaml_path}")
print(yaml_content)

Dataset YAML written → E:\UTS\CNN and Deep Learning\Assignment 3\42028-DLCNN\xbd_yolo\xbd.yaml

path: E:/UTS/CNN and Deep Learning/Assignment 3/42028-DLCNN/xbd_yolo
train: images/train
val:   images/val
test:  images/test

nc: 3
names: ['intact', 'damaged', 'destroyed']



In [ ]:
from ultralytics import YOLO

# Load YOLO11 segmentation model
model = YOLO("yolo11l-seg.pt")   # l=large; use yolo11x-seg.pt for max accuracy

results = model.train(
    data      = yaml_path,
    epochs    = 150,
    imgsz     = IMG_SIZE,
    batch     = 4,
    device    = 0,                    # GPU 0
    project   = os.path.join(YOLO_DS_DIR, "runs"),
    name      = "xbd_seg_v1",
    optimizer = "AdamW",
    lr0       = 1e-4,
    lrf       = 0.01,
    momentum  = 0.937,
    weight_decay = 1e-4,
    warmup_epochs = 3,
    cos_lr    = True,                 # cosine LR schedule
    hsv_h     = 0.015,                # colour augmentation
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    flipud    = 0.5,
    fliplr    = 0.5,
    degrees   = 15.0,                 # rotation aug (satellite imagery)
    translate = 0.1,
    scale     = 0.5,
    val       = True,
    save      = True,
    plots     = True,
    verbose   = True,
)
print("Training complete.")

Ultralytics 8.4.49  Python-3.11.15 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=E:\UTS\CNN and Deep Learning\Assignment 3\42028-DLCNN\xbd_yolo\xbd.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=xbd_seg_v1-2, nbs=64, nms=False, o

KeyboardInterrupt: 

: 

In [ ]:
# Load best weights
best_weights = os.path.join(YOLO_DS_DIR, "runs", "xbd_seg_v1", "weights", "best.pt")
model_inf    = YOLO(best_weights)

# Run prediction on val images
val_img_dir = os.path.join(YOLO_DS_DIR, "images", "val")
val_images  = sorted(glob.glob(os.path.join(val_img_dir, "*.png")))[:50]  # first 50

pred_results = model_inf.predict(
    source  = val_images,
    imgsz   = IMG_SIZE,
    conf    = 0.25,
    iou     = 0.45,
    device  = 0,
    save    = False,   # we'll render manually for heatmaps
    verbose = False,
)
print(f"Inference done on {len(pred_results)} images.")

In [ ]:
def build_damage_heatmap(result, img_size=IMG_SIZE):
    """
    From a YOLO result, build a (H, W, 3) damage heatmap accumulator.
    Returns:
        heatmaps: dict {class_idx: (H,W) float32 accumulation}
        orig_img: (H,W,3) uint8 original image
    """
    h, w = img_size, img_size
    # One heatmap canvas per YOLO class (intact, damaged, destroyed)
    heatmaps = {0: np.zeros((h, w), dtype=np.float32),
                1: np.zeros((h, w), dtype=np.float32),
                2: np.zeros((h, w), dtype=np.float32)}

    orig_img = result.orig_img  # BGR, HxWx3

    if result.masks is None:
        return heatmaps, cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)

    masks      = result.masks.data.cpu().numpy()   # (N, H, W) float32 [0,1]
    class_ids  = result.boxes.cls.cpu().numpy().astype(int)  # (N,)
    confs      = result.boxes.conf.cpu().numpy()             # (N,)

    for mask, cls_id, conf in zip(masks, class_ids, confs):
        # Resize mask to output size if needed
        if mask.shape != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
        # Weight mask by confidence score
        heatmaps[cls_id] += mask * float(conf)

    return heatmaps, cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)


def render_heatmap_figure(orig_img, heatmaps, title="Damage Heatmap"):
    """
    Renders a 5-panel figure:
      [original | intact heatmap | damaged heatmap | destroyed heatmap | composite]
    """
    h, w = orig_img.shape[:2]

    # ── Composite overlay: blend all class heatmaps onto image ───────────────
    composite = orig_img.copy().astype(np.float32)
    # YOLO class 0=intact→green, 1=damaged→orange, 2=destroyed→red
    yolo_colors = {
        0: np.array([0,   200,  0],  dtype=np.float32),  # intact
        1: np.array([255, 165,  0],  dtype=np.float32),  # damaged
        2: np.array([220,  0,   0],  dtype=np.float32),  # destroyed
    }
    for cls_id, color in yolo_colors.items():
        hm = heatmaps[cls_id]
        if hm.max() > 0:
            norm_hm = (hm / hm.max())[:, :, np.newaxis]  # (H,W,1)
            composite = composite * (1 - 0.6 * norm_hm) + color * (0.6 * norm_hm)
    composite = np.clip(composite, 0, 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    axes[0].imshow(orig_img)
    axes[0].set_title("Post-disaster Image"); axes[0].axis("off")

    for i, (cls_id, cmap, label) in enumerate([
        (0, 'Greens',  'Intact'),
        (1, 'Oranges', 'Damaged'),
        (2, 'Reds',    'Destroyed'),
    ]):
        hm = heatmaps[cls_id]
        im = axes[i+1].imshow(hm, cmap=cmap, vmin=0, vmax=max(hm.max(), 1e-6))
        axes[i+1].imshow(orig_img, alpha=0.3)   # ghost image underneath
        axes[i+1].set_title(f"{label} Heatmap"); axes[i+1].axis("off")
        plt.colorbar(im, ax=axes[i+1], fraction=0.046, pad=0.04)

    axes[4].imshow(composite)
    axes[4].set_title("Composite Overlay"); axes[4].axis("off")

    # Legend
    patches = [
        mpatches.Patch(color='green',  label='Intact'),
        mpatches.Patch(color='orange', label='Damaged'),
        mpatches.Patch(color='red',    label='Destroyed'),
    ]
    axes[4].legend(handles=patches, loc='upper right', fontsize=8)

    plt.tight_layout()
    return fig


# ── Render heatmaps for first 6 val predictions ───────────────────────────────
heatmap_dir = os.path.join(YOLO_DS_DIR, "heatmaps")
os.makedirs(heatmap_dir, exist_ok=True)

for i, result in enumerate(pred_results[:6]):
    heatmaps, orig_img = build_damage_heatmap(result, img_size=IMG_SIZE)
    scene_name = os.path.basename(result.path).replace('.png', '')
    fig = render_heatmap_figure(orig_img, heatmaps, title=f"Scene: {scene_name}")
    save_path = os.path.join(heatmap_dir, f"{scene_name}_heatmap.png")
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {save_path}")

In [ ]:
def aggregate_disaster_heatmap(pred_results, img_size=IMG_SIZE):
    """
    Accumulate heatmaps across ALL val predictions to show
    overall spatial damage distribution for the dataset.
    """
    agg = {0: np.zeros((img_size, img_size), dtype=np.float32),
           1: np.zeros((img_size, img_size), dtype=np.float32),
           2: np.zeros((img_size, img_size), dtype=np.float32)}
    count = 0
    for result in tqdm(pred_results, desc="Aggregating heatmaps"):
        heatmaps, _ = build_damage_heatmap(result, img_size)
        for cls_id in agg:
            agg[cls_id] += heatmaps[cls_id]
        count += 1

    # Normalise
    for cls_id in agg:
        if agg[cls_id].max() > 0:
            agg[cls_id] /= agg[cls_id].max()

    # ── Plot aggregate ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle(f"Aggregate Damage Heatmap — {count} Scenes", fontsize=13)

    titles = ['Intact',  'Damaged', 'Destroyed', 'Severity\n(Destroyed − Intact)']
    cmaps  = ['Greens',  'Oranges', 'Reds',      'RdYlGn_r']
    data   = [agg[0],    agg[1],    agg[2],
              np.clip(agg[2] - agg[0], 0, 1)]  # net severity score

    for ax, title, cmap, d in zip(axes, titles, cmaps, data):
        im = ax.imshow(d, cmap=cmap, vmin=0, vmax=1, interpolation='bilinear')
        ax.set_title(title, fontsize=11); ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    agg_path = os.path.join(heatmap_dir, "aggregate_heatmap.png")
    plt.savefig(agg_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Aggregate heatmap saved → {agg_path}")


aggregate_disaster_heatmap(pred_results)

In [ ]:
from ultralytics import YOLO

model_eval = YOLO(best_weights)

# Official YOLO val (gives mAP50, mAP50-95, per-class metrics)
metrics = model_eval.val(
    data    = yaml_path,
    imgsz   = IMG_SIZE,
    batch   = 8,
    device  = 0,
    verbose = True,
    plots   = True,
    save_json = True,
)

print("\n── Segmentation Metrics ──────────────────────────────────────")
print(f"  mAP@50      : {metrics.seg.map50:.4f}")
print(f"  mAP@50-95   : {metrics.seg.map:.4f}")
print(f"\nPer-class mAP@50:")
for cls_name, ap in zip(YOLO_CLASSES, metrics.seg.ap50):
    print(f"  {cls_name:<12s}  {ap:.4f}")